In [ ]:
import numpy as np
import pandas as pd

In [ ]:
books = pd.read_csv('datasets/Books.csv')
ratings = pd.read_csv('datasets/Ratings.csv')
users = pd.read_csv('datasets/Users.csv')

In [ ]:
books.head()

In [ ]:
users.head()

In [ ]:
ratings.head()

In [ ]:
print(books.shape)
print(users.shape)
print(ratings.shape)

In [ ]:
books.isnull().sum()

In [ ]:
ratings.isnull().sum()

In [ ]:
users.isnull().sum()

# Popularity Based Recommendation System

In [ ]:
ratings_with_name = ratings.merge(books, on='ISBN')

In [ ]:
num_rating_df = ratings_with_name.groupby('Book-Title').count()['Book-Rating'].reset_index()
num_rating_df.rename(columns={'Book-Rating': 'num_ratings'}, inplace=True)
num_rating_df

In [ ]:
# avg_rating_df = ratings_with_name.groupby('Book-Title').mean()['Book-Rating'].reset_index()
avg_rating_df = ratings_with_name.groupby("Book-Title")['Book-Rating'].mean().reset_index()
avg_rating_df = avg_rating_df.rename(columns= {'Book-Rating': 'avg_ratings'})
avg_rating_df

In [ ]:
popular_df = num_rating_df.merge(avg_rating_df, on='Book-Title')
popular_df

In [ ]:
popular_df = popular_df[popular_df['num_ratings'] >= 250].sort_values('avg_ratings', ascending=False).head(50)

In [ ]:
popular_df = popular_df.merge(books, on='Book-Title').drop_duplicates('Book-Title')[['Book-Title', 'Book-Author', 'Image-URL-M', 'num_ratings', 'avg_ratings']]

In [ ]:
popular_df

# Collaborative Filtering Based Recommendation System

In [ ]:
x = ratings_with_name.groupby('User-ID').count()['Book-Rating'] > 200
consistent_users = x[x].index
consistent_users

In [ ]:
filtered_ratings = ratings_with_name[ratings_with_name['User-ID'].isin(consistent_users)]

In [ ]:
y  = filtered_ratings.groupby('Book-Title').count()['Book-Rating']>=50
famous_books = y[y].index
famous_books

In [ ]:
final_ratings = filtered_ratings[filtered_ratings['Book-Title'].isin(famous_books)]

In [ ]:
final_ratings

In [ ]:
pt = final_ratings.pivot_table(index='Book-Title', columns='User-ID', values='Book-Rating').fillna(0)
pt

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_scores = cosine_similarity(pt)
similarity_scores.shape

In [ ]:
def recommend(book_name):
    #fetch the index of the book given as input
    index = np.where(pt.index == book_name)[0][0]
    #fetch the similarity scores for all the books with that index
    similar_items = sorted(list(enumerate(similarity_scores[index])), key=lambda x: x[1], reverse=True)[1:6]
    #print the name of similar books
    
    data = []
    for i in similar_items:
        item = []
        # print(pt.index[i[0]])
        temp_df = books[books['Book-Title'] == pt.index[i[0]]]
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Title'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Author'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Image-URL-M'].values))
        
        data.append(item)
    
    return data

In [ ]:
recommend('1984')

In [ ]:
import pickle
pickle.dump(popular_df, open('popular.pkl', 'wb'))
books.drop_duplicates('Book-Title')

In [ ]:
pickle.dump(pt, open('pt.pkl', 'wb'))
pickle.dump(books, open('books.pkl', 'wb'))
pickle.dump(similarity_scores, open('similarity_scores.pkl', 'wb'))